> ## 🔗 Live Deployment: **[https://diamond-price-predicton.streamlit.app/](https://diamond-price-predicton.streamlit.app/)**
> *(Streamlit web app serving the final model, hosted on Streamlit Community Cloud.)*

# Diamond Price Prediction System – 10-Fold Cross Validation (Full Dataset)

| | |
|---|---|
| **Student Name** | Ebbad Ur Rehman |
| **Registration No.** | FA24-BSE-032 |
| **Course** | Machine Learning Fundamentals (AIC354) |
| **Instructor** | Dr. Rao Muhammad Adeel Nawab |
| **Assignment** | 3: Applying, Evaluating, and Comparing Multiple ML Algorithms on a Large Real-World Dataset |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ebbad-dev/Diamond-ML-Problem/blob/main/diamond_price_kfold_10fold.ipynb)

## Setup: Running This Notebook on Google Colab

No installation is needed. Google Colab already has every library this notebook uses.

1. Go to **https://colab.research.google.com** and sign in with a Google account.
2. Click **File → Upload notebook** and choose this `.ipynb` file.
3. **Run 1:** **Runtime → Change runtime type → CPU**, then **Runtime → Run all**. At the end, `cv_results_CPU.csv` is downloaded automatically.
4. **Run 2:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Step 9.6 then compares it with the CPU run automatically.
5. The dataset is downloaded automatically in **Step 2.1** (from Kaggle, or from this project's GitHub repository).

> **Note:** Colab deletes uploaded files when the runtime is reset or disconnected. If you get a *"file not found"* error, run Step 2.1 again.
>
> ⏱ **A full run takes about 10–15 minutes on Colab** (most of it is the MLP neural network and Extra Trees in Step 8).

---

## Introduction

##### **Definition** A regression ML task that predicts the **price (US dollars)** of a diamond from its physical and quality attributes: weight (carat), cut, colour, clarity, proportions (depth, table) and size (x, y, z in mm).
##### **Application** Online jewellery marketplaces, diamond wholesalers and retailers, insurance valuation, pawn-shop and resale pricing, and checking whether a quoted price is fair.

### Aim of This Notebook

This notebook re-uses the methodology of the instructor's *Car Price Prediction – 10-Fold Cross Validation* notebook on a **new, larger problem** and with **5 different algorithms**:

> **Which of 5 regression algorithms predicts diamond prices best, how sure can we be about that answer, and what does each one cost in time and computing resources?**

| | Car Price notebook (instructor) | **This notebook** |
|---|---|---|
| Dataset | Car Features & MSRP (~11,900 rows) | **Kaggle Diamonds (53,940 rows)** |
| Algorithms | Linear Regression, KNN, Decision Tree, Random Forest, Gradient Boosting | **Ridge, Extra Trees, Hist Gradient Boosting, AdaBoost, MLP (neural network)** |
| Evaluation | 10-fold CV, RMSE | 10-fold CV, **RMSE (main)**, MAE and R² |
| Extra | | Execution times, CPU / RAM / GPU monitoring, CPU vs GPU runtime comparison, Streamlit deployment |

### Dataset

**Kaggle:** [Diamonds – shivam2503/diamonds](https://www.kaggle.com/datasets/shivam2503/diamonds), **53,940 diamonds** × 10 attributes (the full dataset is used, not a sample).

| Attribute | Type | Description |
|---|---|---|
| `carat` | numeric | Weight of the diamond (0.2 – 5.01) |
| `cut` | categorical | Quality of the cut: Fair, Good, Very Good, Premium, Ideal |
| `color` | categorical | Colour, from J (worst) to D (best) |
| `clarity` | categorical | How clear it is: I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best) |
| `depth` | numeric | Total depth percentage = 2·z / (x + y) |
| `table` | numeric | Width of the top of the diamond relative to its widest point (%) |
| `x`, `y`, `z` | numeric | Length, width, depth in mm |
| **`price`** | **numeric (target)** | **Price in US dollars ($326 – $18,823)** |

### What is K-Fold Cross Validation?

A single train/test split can be lucky or unlucky. K-Fold Cross Validation removes this luck:

1. Shuffle the data and split it into **K equal parts (folds)**.
2. Train the model on **K−1 folds** and test it on the **remaining fold**.
3. Repeat **K times**, so that **every fold is used as the test set exactly once**.
4. Report the **average** score over the K rounds (and the standard deviation, to show how much it varies).

With **K = 10** on **~53,800 unique diamonds**, each test fold has about **5,380 diamonds** and each training set about **48,400**:

```
Round  1:  [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  2:  [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
   ...
Round 10:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ]
```

### Fair Comparison

Every algorithm is evaluated with:

- the **same 10 folds** (`KFold(n_splits=10, shuffle=True, random_state=42)`),
- the **same pre-processing pipeline**, re-fitted inside each fold (no data leakage),
- the **same evaluation measures**.

### Evaluation Measures

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} \qquad MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i| \qquad R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- **RMSE** (main measure, same as the car notebook): typical error in dollars; large errors are punished more because they are squared. **Lower is better.**
- **MAE**: the average absolute error in dollars; easier to explain to a customer. **Lower is better.**
- **R²**: the fraction of the variation in price that the model explains (1.0 = perfect). **Higher is better.**

# Code – Diamond Price Prediction System
## Step 0: Check the Colab Runtime (CPU, RAM and GPU)

Before training, we record **what hardware this run is using**, so that the execution times in Step 9 can be interpreted correctly.

> **Does a GPU help scikit-learn?** By default, **no**: scikit-learn runs on the **CPU only**, even on a GPU runtime. A GPU is only used if a GPU library takes over the work. Google Colab's GPU runtimes include **RAPIDS cuML**, whose `cuml.accel` mode can run *some* scikit-learn estimators on the GPU without code changes (Step 0.2). In this notebook, that only applies to a small part of the work, which is exactly what this experiment investigates.

In [1]:
import os
import platform
import subprocess

import psutil


def detect_gpu():
    # Returns a description of the NVIDIA GPU, or None if there is no GPU.
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=15)
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout.strip().splitlines()[0]
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return None


GPU_INFO = detect_gpu()
GPU_AVAILABLE = GPU_INFO is not None

print("Runtime hardware")
print("================")
print(f"  Python      : {platform.python_version()}")
print(f"  CPU         : {platform.processor() or platform.machine()}")
print(f"  CPU cores   : {psutil.cpu_count(logical=False)} physical / {psutil.cpu_count(logical=True)} logical")
print(f"  RAM         : {psutil.virtual_memory().total / 1e9:.1f} GB total, "
      f"{psutil.virtual_memory().available / 1e9:.1f} GB available")
print(f"  Disk (free) : {psutil.disk_usage('/').free / 1e9:.1f} GB")
print(f"  GPU         : {GPU_INFO if GPU_AVAILABLE else 'None (CPU runtime)'}")

Runtime hardware
  Python      : 3.13.15
  CPU         : x86_64
  CPU cores   : 1 physical / 2 logical
  RAM         : 13.6 GB total, 12.1 GB available
  Disk (free) : 75.5 GB
  GPU         : Tesla T4, 15360 MiB, 580.82.07


## Step 0.2: (Optional) GPU Acceleration with RAPIDS cuML

Set `USE_CUML_ACCEL = True` **on a GPU runtime** to load `cuml.accel`. It must be loaded **before** scikit-learn is imported, which is why it is here, before Step 1. Estimators that cuML supports then run on the GPU; all others automatically fall back to the normal CPU version.

Leave it `False` for the standard CPU experiment.

In [2]:
USE_CUML_ACCEL = False

CUML_ACCEL_LOADED = False
if USE_CUML_ACCEL and GPU_AVAILABLE:
    try:
        get_ipython().run_line_magic("load_ext", "cuml.accel")
        CUML_ACCEL_LOADED = True
        print("cuml.accel loaded: supported scikit-learn estimators will run on the GPU.")
    except Exception as error:
        print(f"Could not load cuml.accel ({error}). Continuing on the CPU.")
elif USE_CUML_ACCEL:
    print("No GPU found: cuml.accel needs a GPU runtime. Continuing on the CPU.")
else:
    print("cuml.accel not requested: scikit-learn runs on the CPU.")

# A short label for this run, used when saving the timing results
RUNTIME_LABEL = ("GPU+cuML" if CUML_ACCEL_LOADED else "GPU") if GPU_AVAILABLE else "CPU"
print(f"Runtime label for this run: {RUNTIME_LABEL}")

cuml.accel not requested: scikit-learn runs on the CPU.
Runtime label for this run: GPU


## Step 1: Import Libraries

In [3]:
# --- Core Python Libraries ---
import json
import pickle
import threading
import time
import warnings

import numpy as np
import pandas as pd

# --- Machine Learning (scikit-learn) ---
import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.exceptions import ConvergenceWarning

# --- The 5 Regression Algorithms (none of them used in the Car Price notebook) ---
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor

# --- Visualization ---
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

# --- Progress bar (shows elapsed and remaining time) ---
from tqdm import tqdm

# Show large numbers with thousands separators in pandas tables
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print("Library versions used in this run:")
print(f"  NumPy        : {np.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  Matplotlib   : {matplotlib.__version__}")
print(f"  psutil       : {psutil.__version__}")

Library versions used in this run:
  NumPy        : 2.1.3
  Pandas       : 2.2.3
  scikit-learn : 1.6.1
  Matplotlib   : 3.10.0
  psutil       : 5.9.5


# Step 2: Load the Data
## Step 2.1: Get the Dataset (Kaggle → GitHub → upload fallback)

1. If `diamonds.csv` is already in the working folder, it is used directly.
2. Otherwise the notebook downloads the **full dataset from Kaggle** with `kagglehub` (pre-installed in Colab; public datasets need no API key).
3. If that fails, it downloads the same file from this project's GitHub repository.
4. If that also fails, a **"Choose Files"** button appears: select `diamonds.csv`.

In [4]:
DATA_FILE = "diamonds.csv"
KAGGLE_DATASET = "shivam2503/diamonds"
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ebbad-dev/Diamond-ML-Problem/main/diamonds.csv"

# google.colab is only available inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not os.path.exists(DATA_FILE):
    try:
        import shutil
        import kagglehub
        print(f"Downloading '{KAGGLE_DATASET}' from Kaggle ...")
        download_folder = kagglehub.dataset_download(KAGGLE_DATASET)
        shutil.copy(os.path.join(download_folder, DATA_FILE), DATA_FILE)
        print("Download complete.")
    except Exception as error:
        print(f"Kaggle download not available ({error.__class__.__name__}).")

if not os.path.exists(DATA_FILE):
    # Fallback: the same file stored in this project's GitHub repository
    try:
        import urllib.request
        print("Downloading from the project GitHub repository ...")
        urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_FILE)
        print("Download complete.")
    except Exception as error:
        print(f"GitHub download not available ({error.__class__.__name__}).")

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Please click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()

    # If the browser renamed the file (e.g. 'diamonds (1).csv'), rename it back
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready to use ({os.path.getsize(DATA_FILE) / 1e6:.2f} MB).")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Please run this cell again and upload it.")

Using Colab cache for faster access to the 'diamonds' dataset.
Download complete.
'diamonds.csv' is ready to use (3.19 MB).


## Step 2.2: Read the Dataset

This notebook uses **all** the diamonds in the dataset. The Kaggle version of the file has an extra unnamed first column (just the row number), which carries no information, so it is removed.

In [5]:
diamonds = pd.read_csv(DATA_FILE)

# Drop the row-number column that the Kaggle CSV contains ('Unnamed: 0')
index_columns = [c for c in diamonds.columns if c.startswith("Unnamed")]
diamonds = diamonds.drop(columns=index_columns)

print(f"Loaded dataset: {diamonds.shape[0]:,} diamonds, {diamonds.shape[1]} attributes")
print(f"Memory used by the DataFrame: {diamonds.memory_usage(deep=True).sum() / 1e6:.1f} MB")

diamonds.head()

Loaded dataset: 53,940 diamonds, 10 attributes
Memory used by the DataFrame: 11.5 MB


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.50,55.00,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.80,61.00,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.90,65.00,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.40,58.00,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.30,58.00,335,4.34,4.35,2.75


# Step 3: Understand and Pre-process the Data
## Step 3.1: Understand the Data

In [6]:
"""
Purpose of this section:
-------------------------
1. Print the names and types of all attributes (columns).
2. Show the total number of instances (rows).
3. Show how many values are missing in each attribute.
4. Show the values of the categorical attributes.
"""

print("\nAttributes and Types:")
print("=====================\n")
print(diamonds.dtypes.to_string())

print("\nNumber of Instances:")
print("====================\n")
print(f"{len(diamonds):,}")

print("\nMissing Values in Each Attribute:")
print("=================================\n")
missing = diamonds.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

print("\nCategorical Attributes:")
print("=======================\n")
for column in ["cut", "color", "clarity"]:
    print(f"{column:<8}: {diamonds[column].value_counts().to_dict()}")


Attributes and Types:

carat      float64
cut         object
color       object
clarity     object
depth      float64
table      float64
price        int64
x          float64
y          float64
z          float64

Number of Instances:

53,940

Missing Values in Each Attribute:

No missing values

Categorical Attributes:

cut     : {'Ideal': 21551, 'Premium': 13791, 'Very Good': 12082, 'Good': 4906, 'Fair': 1610}
color   : {'G': 11292, 'E': 9797, 'F': 9542, 'H': 8304, 'D': 6775, 'I': 5422, 'J': 2808}
clarity : {'SI1': 13065, 'VS2': 12258, 'SI2': 9194, 'VS1': 8171, 'VVS2': 5066, 'VVS1': 3655, 'IF': 1790, 'I1': 741}


In [7]:
# Summary statistics of the numeric attributes
diamonds.describe().T

,count,mean,std,min,25%,50%,75%,max
carat,"53,940.00",0.80,0.47,0.20,0.40,0.70,1.04,5.01
depth,"53,940.00",61.75,1.43,43.00,61.00,61.80,62.50,79.00
table,"53,940.00",57.46,2.23,43.00,56.00,57.00,59.00,95.00
price,"53,940.00","3,932.80","3,989.44",326.00,950.00,"2,401.00","5,324.25","18,823.00"
x,"53,940.00",5.73,1.12,0.00,4.71,5.70,6.54,10.74
y,"53,940.00",5.73,1.14,0.00,4.72,5.71,6.54,58.90
z,"53,940.00",3.54,0.71,0.00,2.91,3.53,4.04,31.80


### Hidden Missing Values

The table above shows **no missing values**, but the minimum of `x`, `y` and `z` is **0 mm**. A real diamond cannot have zero length, width or depth, so these zeros are really **missing measurements** that were recorded as 0.

In [8]:
zero_size = (diamonds[["x", "y", "z"]] == 0)
print("Diamonds with a size of 0 mm (impossible values):")
print(zero_size.sum().to_string())
print(f"\nRows affected: {zero_size.any(axis=1).sum()} of {len(diamonds):,}")

Diamonds with a size of 0 mm (impossible values):
x     8
y     7
z    20

Rows affected: 20 of 53,940


### Understanding the Output (Price)

Before choosing an evaluation measure, it is important to look at the **range of the output**.

In [9]:
price_summary = diamonds["price"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
price_summary.to_frame(name="Price ($)")

,Price ($)
count,"53,940.00"
mean,"3,932.80"
std,"3,989.44"
min,326.00
25%,950.00
50%,"2,401.00"
75%,"5,324.25"
95%,"13,107.10"
99%,"17,378.22"
max,"18,823.00"


> **Why this matters for RMSE:** prices are **right-skewed**: half of the diamonds cost less than about \$2,400, but the most expensive cost almost \$19,000. Large diamonds are therefore where the biggest dollar errors happen, and RMSE (which squares the errors) is influenced most by them. Unlike the car dataset, there are **no extreme outliers** (the maximum is only about 58× the minimum, compared with a \$2 million supercar in the car data), so RMSE will be much more stable across folds.
>
> As in the car notebook, we predict the price directly **in dollars**, which is easier to understand.

## Step 3.2: Pre-process the Data

In [10]:
"""
Pre-processing steps:
---------------------
1. Remove duplicate rows (the same diamond listed more than once).
   Duplicates are harmful in cross validation: a diamond could appear in BOTH the
   training folds and the test fold, so the RMSE would look better than it really is.
2. Replace the impossible size values (x, y or z = 0 mm) with NaN (missing).
3. These missing values are NOT filled here. They are filled inside the pipeline in
   Step 5, using only the training folds (no data leakage).
"""

rows_before = len(diamonds)
diamonds = diamonds.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(diamonds):,}")
print(f"Instances remaining   : {len(diamonds):,}")

for column in ["x", "y", "z"]:
    diamonds.loc[diamonds[column] == 0, column] = np.nan

print("\nZero sizes replaced with NaN. Missing values now:")
print(diamonds[["x", "y", "z"]].isnull().sum().to_string())

Duplicate rows removed: 146
Instances remaining   : 53,794

Zero sizes replaced with NaN. Missing values now:
x     7
y     6
z    19


# Step 4: Feature Extraction

All **9 input attributes** are used as features: 6 numeric and 3 categorical. Then we separate the **input (X)** from the **output (y)**.

In [11]:
numeric_features = ["carat", "depth", "table", "x", "y", "z"]
categorical_features = ["cut", "color", "clarity"]
target = "price"

# Input features (X) and output (y)
X = diamonds[numeric_features + categorical_features]
y = diamonds[target]

print(f"Input features (X): {X.shape[1]} columns -> {list(X.columns)}")
print(f"Output (y)        : {target}")
print(f"Number of diamonds: {len(X):,}")

X.head()

Input features (X): 9 columns -> ['carat', 'depth', 'table', 'x', 'y', 'z', 'cut', 'color', 'clarity']
Output (y)        : price
Number of diamonds: 53,794


,carat,depth,table,x,y,z,cut,color,clarity
0,0.23,61.50,55.00,3.95,3.98,2.43,Ideal,E,SI2
1,0.21,59.80,61.00,3.89,3.84,2.31,Premium,E,SI1
2,0.23,56.90,65.00,4.05,4.07,2.31,Good,E,VS1
3,0.29,62.40,58.00,4.20,4.23,2.63,Premium,I,VS2
4,0.31,63.30,58.00,4.34,4.35,2.75,Good,J,SI2


# Step 5: Build the Pre-processing Pipeline

The **same** pipeline is used for all 5 algorithms, so that the comparison is fair:

| Feature type | Step | Why |
|---|---|---|
| Numeric | **Fill missing values** with the median | Fills the 0 mm sizes found in Step 3.1 |
| Numeric | **Standard scaling** (mean 0, std 1) | Ridge and the MLP neural network are sensitive to the scale of each feature (e.g. *table* ≈ 57 vs *carat* ≈ 0.8). Tree methods are not affected by scaling |
| Categorical | **One-hot encoding** | Turns `cut`, `color` and `clarity` into 0/1 columns. `sparse_output=False` gives a normal (dense) table, which **Hist Gradient Boosting requires** |

The pre-processing is placed **inside** the pipeline, so in every round of cross validation it is **re-fitted on the training folds only** and then applied to the test fold. This avoids **data leakage**.

`handle_unknown="ignore"` means that if a test fold contains a category that never appeared in the training folds, the model still works.

In [12]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

n_columns_after = preprocessor.fit(X).transform(X.head()).shape[1]
print(f"Pre-processing pipeline created: {X.shape[1]} raw features -> {n_columns_after} model inputs")
preprocessor

Pre-processing pipeline created: 9 raw features -> 26 model inputs


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['carat', 'depth', 'table', 'x', 'y', 'z']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['cut', 'color', 'clarity'])])

# Step 6: Define the 5 Regression Algorithms

None of these algorithms was used in the Car Price Prediction notebook (Linear Regression, KNN, Decision Tree, Random Forest, Gradient Boosting) or in Assignment 2. They cover **four different families** of machine learning:

| # | Algorithm | Family | How it predicts a price |
|---|---|---|---|
| 1 | **Ridge Regression** | Linear | A weighted sum of the features (like Linear Regression), but with an **L2 penalty** that keeps the weights small and makes the model more stable when features are correlated (here `carat`, `x`, `y`, `z` all measure size) |
| 2 | **Extra Trees** (Extremely Randomized Trees) | Ensemble – bagging | Builds 100 trees like Random Forest, but picks the split thresholds **at random**, which makes each tree faster to build and the ensemble less likely to over-fit |
| 3 | **Hist Gradient Boosting** | Ensemble – boosting | Gradient boosting where each feature is first **binned into ≤ 255 bins** (histograms). This makes finding splits much faster on large datasets (the scikit-learn version of LightGBM) |
| 4 | **AdaBoost** | Ensemble – boosting | Trains small trees one after another, giving **more weight to the diamonds that the previous trees predicted badly** |
| 5 | **MLP Regressor** | Neural network | A multi-layer perceptron with two hidden layers (64 and 32 neurons, ReLU) trained with the Adam optimizer |

**Settings:** scikit-learn defaults wherever possible, so the comparison stays simple and fair. Two small exceptions, both needed for this dataset:

- **Extra Trees** uses `n_jobs=-1` (all CPU cores), as Random Forest did in the car notebook.
- **MLP**: the default single hidden layer of 100 neurons with 200 epochs does not converge on prices in dollars. We use `(64, 32)` layers, up to 300 epochs, and `early_stopping=True` (it holds out 10% of the *training* fold and stops when that score stops improving, which is fair because it never sees the test fold).

`random_state=42` makes all results reproducible.

In [13]:
algorithms = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=50, random_state=42),
    "MLP Neural Network": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300,
                                       early_stopping=True, random_state=42),
}

print("Algorithms to compare:")
for number, (name, algorithm) in enumerate(algorithms.items(), start=1):
    print(f"  {number}. {name:<24} {algorithm}")

Algorithms to compare:
  1. Ridge Regression         Ridge()
  2. Extra Trees              ExtraTreesRegressor(n_jobs=-1, random_state=42)
  3. Hist Gradient Boosting   HistGradientBoostingRegressor(random_state=42)
  4. AdaBoost                 AdaBoostRegressor(random_state=42)
  5. MLP Neural Network       MLPRegressor(early_stopping=True, hidden_layer_sizes=(64, 32), max_iter=300,
             random_state=42)


# Step 7: Set Up 10-Fold Cross Validation
## Step 7.1: Create the K-Fold Splitter

- `n_splits=10`: split the data into 10 folds
- `shuffle=True`: shuffle first. The original file is **sorted by price**, so without shuffling the first fold would contain only the cheapest diamonds and the last fold only the most expensive ones.
- `random_state=42`: the **same folds are used for all 5 algorithms**, so every algorithm is tested on exactly the same diamonds.

In [14]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)
folds = list(kfold.split(X))     # computed once and re-used for every algorithm

print(f"{K}-Fold Cross Validation created (shuffle=True, random_state=42)")

10-Fold Cross Validation created (shuffle=True, random_state=42)


## Step 7.2: Inspect the Folds

In [15]:
fold_sizes = []
for fold_number, (train_index, test_index) in enumerate(folds, start=1):
    fold_sizes.append({
        "Fold (Round)": fold_number,
        "Training Diamonds": len(train_index),
        "Testing Diamonds": len(test_index),
        "Avg Price of Test Diamonds ($)": y.iloc[test_index].mean(),
        "Most Expensive Test Diamond ($)": y.iloc[test_index].max(),
    })

fold_sizes_table = pd.DataFrame(fold_sizes).set_index("Fold (Round)")
fold_sizes_table

,Training Diamonds,Testing Diamonds,Avg Price of Test Diamonds ($),Most Expensive Test Diamond ($)
Fold (Round),,,,
1,48414,5380,"3,862.60",18791
2,48414,5380,"3,908.42",18779
3,48414,5380,"3,875.83",18818
4,48414,5380,"3,965.32",18823
5,48415,5379,"3,922.57",18787
6,48415,5379,"3,921.42",18797
7,48415,5379,"3,914.45",18803
8,48415,5379,"4,046.61",18795
9,48415,5379,"3,862.48",18795


Because the data is large and shuffled, every fold has almost the **same average price** (≈ \$3,900) and a very similar most expensive diamond. The folds are therefore very similar to each other, so we expect a **small standard deviation** of the RMSE across folds.

# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms

For each algorithm we build a pipeline (**pre-processing → algorithm**). Then, for each of the 10 folds, we:

1. make a **fresh copy** of the pipeline (`clone`), so nothing learned in one round is carried into the next,
2. **train** it on the training folds (and time it: *fit time*),
3. **predict** the prices of the diamonds in the test fold (and time it: *predict time*),
4. calculate **RMSE, MAE and R²** for that fold.

### Monitoring Resource Usage

While each algorithm runs, a small **background thread** records the following every half second:

- **CPU usage (%)** of the whole machine (100% = all cores busy),
- **RAM used** by this notebook (in GB),
- **GPU utilisation (%) and GPU memory** (only on a GPU runtime, read with `nvidia-smi`).

### Progress Bar (tqdm)

There is one step for each algorithm on each fold, so the bar has **5 × 10 = 50 steps**:

```
Extra Trees, fold 4/10:  28%|██▊       | 14/50 [00:52<02:41,  4.48s/fold]
```

| Part of the bar | Meaning |
|---|---|
| `Extra Trees, fold 4/10` | The algorithm and fold being trained right now |
| `28%` and `14/50` | How much of the work is done |
| `00:52` (before `<`) | **Elapsed time** |
| `02:41` (after `<`) | **Estimated remaining time** |
| `4.48s/fold` | Speed (seconds per fold) |

> ⏱ **This step takes about 5–15 minutes**, depending on the runtime. The **MLP** is the slowest, followed by Extra Trees.

In [16]:
def read_gpu_usage():
    # Returns (GPU utilisation %, GPU memory used in GB) from nvidia-smi
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=5)
    utilisation, memory_mb = result.stdout.strip().splitlines()[0].split(",")
    return float(utilisation), float(memory_mb) / 1024


class ResourceMonitor:
    # Records CPU %, RAM and GPU usage in a background thread while a block of code runs.

    def __init__(self, interval=0.5):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

    def __enter__(self):
        self.cpu_percent, self.ram_gb, self.gpu_percent, self.gpu_memory_gb = [], [], [], []
        self._stop = threading.Event()
        psutil.cpu_percent(None)                      # start a new CPU measurement window
        self._thread = threading.Thread(target=self._record, daemon=True)
        self._thread.start()
        return self

    def _record(self):
        while not self._stop.wait(self.interval):
            self.cpu_percent.append(psutil.cpu_percent(None))
            self.ram_gb.append(self.process.memory_info().rss / 1e9)
            if GPU_AVAILABLE:
                try:
                    utilisation, memory_gb = read_gpu_usage()
                    self.gpu_percent.append(utilisation)
                    self.gpu_memory_gb.append(memory_gb)
                except Exception:
                    pass

    def __exit__(self, *exc_info):
        self._stop.set()
        self._thread.join()

    def summary(self):
        def mean(values):
            return float(np.mean(values)) if values else np.nan

        def peak(values):
            return float(np.max(values)) if values else np.nan

        return {
            "Avg CPU (%)": mean(self.cpu_percent),
            "Peak RAM (GB)": peak(self.ram_gb),
            "Avg GPU (%)": mean(self.gpu_percent),
            "Peak GPU Mem (GB)": peak(self.gpu_memory_gb),
        }


print("Resource monitor ready" + (" (CPU, RAM and GPU)." if GPU_AVAILABLE else " (CPU and RAM; no GPU)."))

Resource monitor ready (CPU, RAM and GPU).


In [ ]:
fold_results = []      # one row per (algorithm, fold)
resource_usage = {}    # CPU / RAM / GPU usage per algorithm
total_time = {}        # total time for all 10 folds (fit + predict)

total_steps = len(algorithms) * K

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)

    with tqdm(total=total_steps, unit="fold") as progress_bar:
        for name, algorithm in algorithms.items():
            model = Pipeline([
                ("preprocessor", preprocessor),
                ("regressor", algorithm),
            ])

            start = time.perf_counter()
            with ResourceMonitor() as monitor:
                for fold_number, (train_index, test_index) in enumerate(folds, start=1):
                    progress_bar.set_description(f"{name}, fold {fold_number}/{K}")

                    fold_model = clone(model)                                   # fresh, untrained copy

                    fit_start = time.perf_counter()
                    fold_model.fit(X.iloc[train_index], y.iloc[train_index])    # train on K-1 folds
                    fit_seconds = time.perf_counter() - fit_start

                    predict_start = time.perf_counter()
                    predictions = fold_model.predict(X.iloc[test_index])        # test on 1 fold
                    predict_seconds = time.perf_counter() - predict_start

                    actual = y.iloc[test_index]
                    fold_results.append({
                        "Algorithm": name,
                        "Fold": fold_number,
                        "RMSE": root_mean_squared_error(actual, predictions),
                        "MAE": mean_absolute_error(actual, predictions),
                        "R2": r2_score(actual, predictions),
                        "Fit Time (s)": fit_seconds,
                        "Predict Time (s)": predict_seconds,
                    })
                    progress_bar.update(1)

            total_time[name] = time.perf_counter() - start
            resource_usage[name] = monitor.summary()

            algorithm_rmse = [r["RMSE"] for r in fold_results if r["Algorithm"] == name]
            tqdm.write(f"{name:<24} -> mean RMSE = ${np.mean(algorithm_rmse):>8,.2f}   "
                       f"({total_time[name]:6.1f} s, peak RAM {resource_usage[name]['Peak RAM (GB)']:.2f} GB)")

        progress_bar.set_description("Done")

fold_results = pd.DataFrame(fold_results)
print(f"\n{K}-fold cross validation completed for all {len(algorithms)} algorithms "
      f"in {sum(total_time.values()) / 60:.1f} minutes.")

Extra Trees, fold 1/10:  20%|██        | 10/50 [00:01<00:07,  5.44fold/s]      

Ridge Regression         -> mean RMSE = $1,136.38   (   1.8 s, peak RAM 0.34 GB)


Extra Trees, fold 3/10:  24%|██▍       | 12/50 [00:36<05:39,  8.93s/fold]

# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

Each row is one round of cross validation and each column is one algorithm. The **lowest RMSE in each row** (the best algorithm on that fold) is highlighted in green.

In [ ]:
per_fold_table = fold_results.pivot(index="Fold", columns="Algorithm", values="RMSE")[list(algorithms)]
per_fold_table.index = [f"Fold {i}" for i in per_fold_table.index]
per_fold_table.index.name = "Fold"

(per_fold_table.style
    .format("${:,.0f}")
    .highlight_min(axis=1, props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

In [ ]:
folds_won = per_fold_table.idxmin(axis=1).value_counts().reindex(list(algorithms), fill_value=0)
print("Number of folds won (lowest RMSE on that fold):")
print(folds_won.to_string())

## Step 9.2: Summary Comparison Table

This is the main result of the notebook. For each algorithm:

| Column | Meaning |
|---|---|
| **Mean RMSE** | Average RMSE over the 10 folds. This is the **main measure for ranking** the algorithms (lower is better) |
| **Std RMSE** | Standard deviation over the folds. A **small value means the algorithm is consistent** |
| **Best / Worst Fold** | Lowest and highest RMSE across the folds |
| **Mean MAE** | Average absolute error in dollars |
| **Mean R²** | Fraction of the price variation explained (1.0 = perfect) |
| **RMSE as % of Avg Price** | Mean RMSE divided by the average diamond price, to put the error in context |
| **Time (s)** | Total time to train and test across all 10 folds |
| **Rank** | 1 = lowest mean RMSE = best algorithm |

In [ ]:
average_price = y.mean()
by_algorithm = fold_results.groupby("Algorithm")

summary = pd.DataFrame({
    "Mean RMSE ($)": per_fold_table.mean(),
    "Std RMSE ($)": per_fold_table.std(),
    "Best Fold RMSE ($)": per_fold_table.min(),
    "Worst Fold RMSE ($)": per_fold_table.max(),
    "Mean MAE ($)": by_algorithm["MAE"].mean(),
    "Mean R2": by_algorithm["R2"].mean(),
    "RMSE as % of Avg Price": per_fold_table.mean() / average_price * 100,
    "Time (s)": pd.Series(total_time),
})
summary = summary.sort_values("Mean RMSE ($)")
summary.insert(0, "Rank", range(1, len(summary) + 1))
summary.index.name = "Algorithm"

print(f"Average diamond price in this data: ${average_price:,.2f}\n")

money_columns = ["Mean RMSE ($)", "Std RMSE ($)", "Best Fold RMSE ($)", "Worst Fold RMSE ($)", "Mean MAE ($)"]
(summary.style
    .format({**{c: "${:,.0f}" for c in money_columns},
             "Mean R2": "{:.4f}",
             "RMSE as % of Avg Price": "{:.1f}%",
             "Time (s)": "{:.1f}"})
    .highlight_min(subset=["Mean RMSE ($)", "Mean MAE ($)", "Time (s)"],
                   props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;")
    .highlight_max(subset=["Mean R2"], props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))

## Step 9.3: Visual Comparison

**Chart 1** shows the mean RMSE of each algorithm (shorter bar = better). The black whisker shows ± 1 standard deviation across the 10 folds.

**Chart 2** shows the RMSE of **every individual fold** as a dot, so you can see how spread out each algorithm's results are.

In [ ]:
# Chart colours: one blue for the data, dark/grey text, light grid
BAR_COLOR = "#2a78d6"
TEXT_COLOR = "#0b0b0b"
MUTED_TEXT = "#52514e"
GRID_COLOR = "#e4e3df"
dollars = FuncFormatter(lambda value, _: f"${value:,.0f}")

def style_axis(ax, formatter=dollars):
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(colors=MUTED_TEXT, length=0)
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)

order = summary.index.tolist()            # best algorithm first
positions = np.arange(len(order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2))

# ---- Chart 1: mean RMSE with +/- 1 std ----
means = summary["Mean RMSE ($)"]
stds = summary["Std RMSE ($)"]
ax1.barh(positions, means, height=0.5, color=BAR_COLOR,
         xerr=stds, error_kw=dict(ecolor=TEXT_COLOR, elinewidth=1.2, capsize=4))
for i, name in enumerate(order):
    label = f"${means[name]:,.0f}" + ("   (best)" if i == 0 else "")
    ax1.text(means[name] + stds[name] + means.max() * 0.02, i, label,
             va="center", color=TEXT_COLOR, fontsize=10)
ax1.set_yticks(positions, order, color=TEXT_COLOR)
ax1.invert_yaxis()
ax1.set_xlim(0, (means + stds).max() * 1.3)
style_axis(ax1)
ax1.set_title("Mean RMSE across 10 folds (lower is better)", loc="left", color=TEXT_COLOR, fontsize=12)
ax1.set_xlabel("RMSE (US dollars)", color=MUTED_TEXT)

# ---- Chart 2: RMSE of every fold ----
for i, name in enumerate(order):
    ax2.scatter(per_fold_table[name], np.full(K, i), s=60, color=BAR_COLOR,
                edgecolor="white", linewidth=1.5, zorder=3,
                label="RMSE of one fold" if i == 0 else None)
    ax2.scatter(means[name], i, marker="|", s=500, color=TEXT_COLOR, linewidth=2.5, zorder=4,
                label="Mean RMSE" if i == 0 else None)
ax2.set_yticks(positions, order, color=TEXT_COLOR)
ax2.invert_yaxis()
ax2.set_xlim(0, per_fold_table.values.max() * 1.1)
style_axis(ax2)
ax2.set_title("RMSE of each fold (spread = consistency)", loc="left", color=TEXT_COLOR, fontsize=12)
ax2.set_xlabel("RMSE (US dollars)", color=MUTED_TEXT)
ax2.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, labelcolor=MUTED_TEXT)

plt.tight_layout()
plt.show()

## Step 9.4: Execution Time and Resource Usage

Accuracy is not the only thing that matters. This table shows **how long** each algorithm took and **how much** of the machine it used:

| Column | Meaning |
|---|---|
| **Total Time** | Training + testing over all 10 folds |
| **Avg Fit / Predict per Fold** | Average time to train on ~48,400 diamonds / predict ~5,380 diamonds |
| **Avg CPU (%)** | Average CPU usage of the whole machine while the algorithm ran. On a 2-core Colab machine, ≈ 50% means **one core** was busy, ≈ 100% means **both** |
| **Peak RAM (GB)** | Highest memory used by this notebook while the algorithm ran |
| **Avg GPU (%) / Peak GPU Mem** | GPU usage (only filled on a GPU runtime) |

In [ ]:
timing = pd.DataFrame({
    "Total Time (s)": pd.Series(total_time),
    "Avg Fit per Fold (s)": by_algorithm["Fit Time (s)"].mean(),
    "Avg Predict per Fold (s)": by_algorithm["Predict Time (s)"].mean(),
    "Predictions per Second": len(y) / K / by_algorithm["Predict Time (s)"].mean(),
}).join(pd.DataFrame(resource_usage).T)
timing = timing.loc[order]
timing.index.name = "Algorithm"

if not GPU_AVAILABLE:
    timing = timing.drop(columns=["Avg GPU (%)", "Peak GPU Mem (GB)"])

print(f"Runtime: {RUNTIME_LABEL}  |  CPU cores: {psutil.cpu_count(logical=True)}  |  "
      f"GPU: {GPU_INFO if GPU_AVAILABLE else 'none'}\n")
timing.style.format({
    "Total Time (s)": "{:.1f}", "Avg Fit per Fold (s)": "{:.2f}", "Avg Predict per Fold (s)": "{:.3f}",
    "Predictions per Second": "{:,.0f}", "Avg CPU (%)": "{:.0f}", "Peak RAM (GB)": "{:.2f}",
    "Avg GPU (%)": "{:.0f}", "Peak GPU Mem (GB)": "{:.2f}",
})

In [ ]:
# ---- Chart 3: accuracy vs. training time ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(summary["Time (s)"], summary["Mean RMSE ($)"], s=90, color=BAR_COLOR,
           edgecolor="white", linewidth=1.5, zorder=3)
for name in order:
    ax.annotate(name, (summary.loc[name, "Time (s)"], summary.loc[name, "Mean RMSE ($)"]),
                textcoords="offset points", xytext=(8, 6), color=TEXT_COLOR, fontsize=10)
ax.set_xscale("log")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID_COLOR)
ax.spines["left"].set_color(GRID_COLOR)
ax.tick_params(colors=MUTED_TEXT, length=0)
ax.yaxis.set_major_formatter(dollars)
ax.grid(color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel("Total time for 10 folds (seconds, log scale)", color=MUTED_TEXT)
ax.set_ylabel("Mean RMSE (US dollars)", color=MUTED_TEXT)
ax.set_title("Accuracy vs. time (bottom-left is best)", loc="left", color=TEXT_COLOR, fontsize=12)
plt.tight_layout()
plt.show()

## Step 9.5: GPU Investigation – Save This Run's Results

To find out whether a GPU helps, run the notebook **twice** on Colab and compare:

| Run | Colab runtime type | `USE_CUML_ACCEL` (Step 0.2) | Results file |
|---|---|---|---|
| 1 | **CPU** | `False` | `cv_results_CPU.csv` |
| 2 | **T4 GPU** | `False` (scikit-learn ignores the GPU) | `cv_results_GPU.csv` |
| 3 *(optional)* | **T4 GPU** | `True` (cuML takes over supported estimators) | `cv_results_GPU+cuML.csv` |

The cell below saves this run's timing and accuracy results. **On Colab the file is downloaded to your computer automatically** (allow the download if the browser asks), because Colab deletes it when the runtime is reset.

In [ ]:
RESULTS_FILE = f"cv_results_{RUNTIME_LABEL}.csv"
DOWNLOAD_RESULTS = IN_COLAB     # on Colab the file is downloaded automatically

run_results = summary[["Mean RMSE ($)", "Std RMSE ($)", "Mean MAE ($)", "Mean R2", "Time (s)"]].join(
    timing.drop(columns=["Total Time (s)"]))
run_results.insert(0, "Runtime", RUNTIME_LABEL)
run_results.insert(1, "Hardware", GPU_INFO if GPU_AVAILABLE else f"{psutil.cpu_count()} CPU cores")
run_results.to_csv(RESULTS_FILE)
print(f"Results of this run saved to '{RESULTS_FILE}'")

if IN_COLAB and DOWNLOAD_RESULTS:
    files.download(RESULTS_FILE)

## Step 9.6: Compare CPU and GPU Runtimes

On a **Colab GPU run** this cell runs automatically: it loads the results of the **Colab CPU run** (`cv_results_CPU.csv`, stored in the project's GitHub repository, or uploaded if that is not available), then shows the total time of each algorithm on each runtime and the **speed-up** (CPU time ÷ GPU time; a value close to 1.0× means the GPU did **not** help). It also prints a ready-to-paste table for the section below.

In [ ]:
COMPARE_RUNTIMES = IN_COLAB and GPU_AVAILABLE   # automatic on a Colab GPU run

CPU_RESULTS_URL = "https://raw.githubusercontent.com/ebbad-dev/Diamond-ML-Problem/main/cv_results_CPU.csv"

if COMPARE_RUNTIMES:
    if not os.path.exists("cv_results_CPU.csv"):
        # The CPU run's results are stored in the project GitHub repository
        try:
            import urllib.request
            urllib.request.urlretrieve(CPU_RESULTS_URL, "cv_results_CPU.csv")
            print("Loaded the Colab CPU run's results from GitHub.")
        except Exception as error:
            print(f"Could not download the CPU results ({error.__class__.__name__}).")
    if IN_COLAB and not os.path.exists("cv_results_CPU.csv"):
        print("Upload cv_results_CPU.csv (downloaded by your CPU run) - or click Cancel to skip:")
        try:
            files.upload()
        except Exception:
            print("No file uploaded.")

    result_files = sorted(f for f in os.listdir(".") if f.startswith("cv_results_") and f.endswith(".csv"))
    all_runs = pd.concat([pd.read_csv(f, index_col="Algorithm") for f in result_files])
    time_by_runtime = all_runs.pivot_table(index="Algorithm", columns="Runtime", values="Time (s)")
    time_by_runtime = time_by_runtime.loc[[a for a in order if a in time_by_runtime.index]]

    if "CPU" in time_by_runtime.columns:
        for runtime in time_by_runtime.columns.drop("CPU"):
            time_by_runtime[f"Speed-up {runtime} vs CPU"] = time_by_runtime["CPU"] / time_by_runtime[runtime]

    print(f"Total 10-fold time (seconds) per runtime, from: {result_files}\n")
    display(time_by_runtime.style.format("{:.2f}"))

    # Ready-to-paste Markdown table for "Recorded Colab Execution Times" below
    runtimes = [r for r in ["CPU", "GPU", "GPU+cuML"] if r in time_by_runtime.columns]
    lines = ["| Algorithm | " + " | ".join(f"Colab {r} (s)" for r in runtimes) + " | Did the GPU help? |",
             "|---" * (len(runtimes) + 2) + "|"]
    for name, row in time_by_runtime.iterrows():
        speedups = [row[c] for c in time_by_runtime.columns if c.startswith("Speed-up")]
        helped = ("yes" if max(speedups) >= 1.2 else "no") if speedups else ""
        lines.append(f"| {name} | " + " | ".join(f"{row[r]:.1f}" for r in runtimes) + f" | {helped} |")
    print("\nCopy this table into 'Recorded Colab Execution Times':\n")
    print("\n".join(lines))
else:
    print("Runs automatically on a Colab GPU runtime (after the CPU run).")

### Recorded Colab Execution Times

*Fill in this table from your own Colab runs (Step 9.4 / 9.6). Times are the total for all 10 folds, in seconds.*

| Algorithm | Colab CPU runtime | Colab T4 GPU runtime | Colab T4 GPU + cuML | Did the GPU help? |
|---|---|---|---|---|
| Ridge Regression | | | | |
| Extra Trees | | | | |
| Hist Gradient Boosting | | | | |
| AdaBoost | | | | |
| MLP Neural Network | | | | |
| **Total** | | | | |

## Step 9.7: Observations

> The numbers below come from our run (4-core CPU, no GPU, scikit-learn 1.9.1). The RMSE values are reproducible (`random_state=42`); **times depend on the machine**, and Colab's free runtime (2 vCPUs) is usually slower, but the overall pattern should be the same.

1. **Extra Trees has the lowest mean RMSE (≈ \$543 ± \$20)**, with **Hist Gradient Boosting a close second (≈ \$551 ± \$27)**. The difference (≈ \$8, 1.5%) is small, but it is **consistent**: Extra Trees wins **7 of the 10 folds**, and the gap is larger than one standard error (≈ \$6), so Extra Trees is genuinely (slightly) the most accurate. Its MAE is also lower (≈ \$266 vs ≈ \$290).
2. **Tree ensembles clearly win.** Both explain **≈ 98% of the price variation (R² ≈ 0.98)**, and the typical error is only ≈ 14% of the average price (≈ \$3,930). Diamond prices are **not linear**: price grows faster than carat, and each step in cut, colour or clarity multiplies the price rather than adding a fixed amount. That is why **Ridge Regression** (a weighted sum) has **twice the error** (≈ \$1,136, R² ≈ 0.92).
3. **AdaBoost performs worst (≈ \$1,403).** With default settings it combines only 50 **very shallow trees (depth 3)**, so it can only predict a few dozen different price levels. Stronger base trees or more estimators would help, but would also make it slower.
4. **The MLP neural network is third (≈ \$682) but the least consistent (std ≈ \$144).** On 9 folds its RMSE is ≈ \$600 – \$680, but on **fold 2 it jumps to ≈ \$1,087**: neural-network training depends on the random starting weights and where early stopping halts, so one training run can be much worse than the others. Tree methods do not have this problem.
5. **The folds are very consistent** (std ≈ \$20 – \$36 for most algorithms, i.e. about 4% of the RMSE). Compare this with the car dataset, where the standard deviation was almost as large as the RMSE because of a few \$1–2 million supercars. A large dataset **without extreme outliers** gives a much more reliable comparison.
6. **Accuracy vs. speed:** Ridge took ≈ 2 s, Hist Gradient Boosting ≈ 10 s, AdaBoost ≈ 45 s, Extra Trees ≈ 60 s and the MLP ≈ 265 s (**about 70% of the total time**, for only the third-best accuracy). Hist Gradient Boosting is ≈ **6× faster** than Extra Trees for only 1.5% more RMSE, thanks to binning the features into histograms.
7. **Resource usage:** Extra Trees (`n_jobs=-1`) and Hist Gradient Boosting (multi-threaded with OpenMP) kept **almost all CPU cores busy** (≈ 95% and ≈ 82%), while AdaBoost used only **one core** (≈ 26% of 4 cores). Extra Trees also needed the most RAM (≈ 1.1 GB peak), because it stores 100 fully-grown trees; all other algorithms stayed under 0.8 GB.
8. **GPU:** scikit-learn runs **only on the CPU**, so selecting a T4 GPU runtime does not speed up any of these 5 algorithms (the GPU stays at ≈ 0% utilisation). With `cuml.accel`, cuML only takes over the estimators it supports; among our five that is **Ridge Regression**, which already takes ≈ 2 s, so moving the data to the GPU can cost more than it saves. Extra Trees, Hist Gradient Boosting, AdaBoost and the MLP fall back to the CPU. **Record your own Colab CPU and GPU times in the table above** to confirm this.

# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm (Statistical Choice)

**Cross validation is used to *choose* an algorithm, not to produce the final model.** The 50 models built during cross validation were only used to measure performance and are now thrown away.

The car notebook simply picked the algorithm with the lowest mean RMSE. A small difference in mean RMSE can be just noise, so we use the well-known **one-standard-error rule**:

1. Find the algorithm with the lowest mean RMSE.
2. Compute its **standard error** = std RMSE / √K. Any algorithm whose mean RMSE is within one standard error of the best is **statistically as good** as the best.
3. Among those, prefer the **fastest** one (cheaper and quicker to serve).

If no other algorithm is close enough, the rule simply picks the algorithm with the lowest RMSE.

In [ ]:
lowest_rmse_name = summary.index[0]
standard_error = summary.loc[lowest_rmse_name, "Std RMSE ($)"] / np.sqrt(K)
threshold = summary.loc[lowest_rmse_name, "Mean RMSE ($)"] + standard_error

within_one_se = summary[summary["Mean RMSE ($)"] <= threshold]
statistical_choice = within_one_se["Time (s)"].idxmin()

print(f"Lowest mean RMSE    : {lowest_rmse_name} (${summary.loc[lowest_rmse_name, 'Mean RMSE ($)']:,.2f})")
print(f"Standard error      : ${standard_error:,.2f}  ->  threshold = ${threshold:,.2f}")
print(f"Within 1 SE         : {list(within_one_se.index)}")
print(f"Statistical choice  : {statistical_choice}")

## Step 10.2: Train on All the Data and Check That the Model Can Be Deployed

Now we train the chosen algorithm on **all diamonds**. There is no separate test set here, because the cross-validation results from Step 9 are already our estimate of how well the model will perform on new, unseen diamonds.

The model must also work **inside the web app**. Two practical limits apply:

- **GitHub** (where Streamlit Community Cloud reads the app from) rejects files larger than **100 MB**.
- A **Streamlit Community Cloud** app has only about **1 GB of RAM**, and the whole model must be loaded into memory.

Tree ensembles such as Extra Trees grow every tree until each leaf holds one diamond, so with ~54,000 diamonds and 100 trees the saved model can be **hundreds of MB**. The cell below therefore tries the algorithms **in order of preference** (the statistical choice first, then the rest by mean RMSE), measures the size of each trained model, and **deploys the first one that fits within 100 MB**.

In [ ]:
MAX_MODEL_MB = 100

preference_order = [statistical_choice] + [name for name in summary.index if name != statistical_choice]
size_check = []

for name in preference_order:
    candidate_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", clone(algorithms[name])),
    ])
    start = time.perf_counter()
    candidate_model.fit(X, y)
    training_seconds = time.perf_counter() - start
    size_mb = len(pickle.dumps(candidate_model)) / 1e6

    fits = size_mb <= MAX_MODEL_MB
    size_check.append({"Algorithm": name,
                       "Mean RMSE ($)": summary.loc[name, "Mean RMSE ($)"],
                       "Training Time on All Data (s)": training_seconds,
                       "Model Size (MB)": size_mb,
                       f"Fits {MAX_MODEL_MB} MB limit": "yes" if fits else "NO"})
    if fits:
        best_algorithm_name, final_model, final_training_seconds = name, candidate_model, training_seconds
        break
    del candidate_model

best_mean_rmse = summary.loc[best_algorithm_name, "Mean RMSE ($)"]
best_std_rmse = summary.loc[best_algorithm_name, "Std RMSE ($)"]
best_mae = summary.loc[best_algorithm_name, "Mean MAE ($)"]
best_r2 = summary.loc[best_algorithm_name, "Mean R2"]

display(pd.DataFrame(size_check).set_index("Algorithm").style.format(
    {"Mean RMSE ($)": "${:,.0f}", "Training Time on All Data (s)": "{:.1f}", "Model Size (MB)": "{:,.2f}"}))

print(f"\nDeployed algorithm: {best_algorithm_name}  (trained on {len(X):,} diamonds in {final_training_seconds:.1f} s)")
print(f"Expected RMSE     : ${best_mean_rmse:,.2f} (+/- ${best_std_rmse:,.2f})")
print(f"Expected MAE      : ${best_mae:,.2f}   |   R2 = {best_r2:.4f}")
if best_algorithm_name != lowest_rmse_name:
    extra = best_mean_rmse - summary.loc[lowest_rmse_name, "Mean RMSE ($)"]
    print(f"Cost of deployability: RMSE is ${extra:,.2f} "
          f"({extra / summary.loc[lowest_rmse_name, 'Mean RMSE ($)'] * 100:.1f}%) higher than {lowest_rmse_name}.")

## Step 10.3: Save the Trained Model (and Its Information for the Web App)

Two files are saved:

- **`diamond_price_model.pkl`**: the **whole pipeline** (imputer, scaler, one-hot encoder **and** the algorithm), so it can predict directly from raw diamond attributes.
- **`model_info.json`**: information that the **Streamlit app** displays: the algorithm, its cross-validation errors, the comparison of all 5 algorithms, the allowed category values and the ranges of each feature.

> **Important for deployment:** a pickled model should be loaded with the **same scikit-learn version** that created it. The cell prints the exact line to put in the app's `requirements.txt`.

To download the files in Colab, set `DOWNLOAD_MODEL = True` and run the cell again, or use the **Files** panel (folder icon) on the left.

In [ ]:
MODEL_FILE = "diamond_price_model.pkl"
INFO_FILE = "model_info.json"
DOWNLOAD_MODEL = False

with open(MODEL_FILE, "wb") as f:
    pickle.dump(final_model, f)

model_info = {
    "problem": "Diamond Price Prediction",
    "dataset": f"Kaggle {KAGGLE_DATASET} ({len(X):,} unique diamonds)",
    "algorithm": best_algorithm_name,
    "scikit_learn_version": sklearn.__version__,
    "cross_validation": f"{K}-fold (shuffle=True, random_state=42)",
    "cv_rmse": float(best_mean_rmse),
    "cv_rmse_std": float(best_std_rmse),
    "cv_mae": float(best_mae),
    "cv_r2": float(best_r2),
    "selection_note": (
        f"{lowest_rmse_name} had the lowest cross-validation RMSE, but its trained model "
        f"({size_check[0]['Model Size (MB)']:,.0f} MB) is too large to deploy (limit {MAX_MODEL_MB} MB). "
        f"{best_algorithm_name} was deployed instead: its model is {size_check[-1]['Model Size (MB)']:.2f} MB "
        f"and its RMSE is only {(best_mean_rmse / summary.loc[lowest_rmse_name, 'Mean RMSE ($)'] - 1) * 100:.1f}% higher."
        if best_algorithm_name != statistical_choice else
        f"{best_algorithm_name} was selected with the one-standard-error rule (the fastest algorithm whose "
        f"mean RMSE is within one standard error of the best) and fits the {MAX_MODEL_MB} MB deployment limit."
    ),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "categories": {
        "cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
        "color": ["J", "I", "H", "G", "F", "E", "D"],
        "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
    },
    "numeric_ranges": {c: {"min": float(X[c].min()), "max": float(X[c].max()),
                           "median": float(X[c].median())} for c in numeric_features},
    "comparison": summary.reset_index()[["Rank", "Algorithm", "Mean RMSE ($)", "Std RMSE ($)",
                                        "Mean MAE ($)", "Mean R2", "Time (s)"]].to_dict(orient="records"),
}
with open(INFO_FILE, "w") as f:
    json.dump(model_info, f, indent=2)

print(f"Trained model saved as '{MODEL_FILE}' ({os.path.getsize(MODEL_FILE) / 1e6:.2f} MB)")
print(f"Model information saved as '{INFO_FILE}'")
print(f"\nrequirements.txt line for the web app:  scikit-learn=={sklearn.__version__}")

if os.path.getsize(MODEL_FILE) > 100e6:
    print("WARNING: the model is larger than 100 MB, which is too large for GitHub / Streamlit Cloud.")

if IN_COLAB and DOWNLOAD_MODEL:
    files.download(MODEL_FILE)
    files.download(INFO_FILE)

# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

By default the notebook uses the example diamond below, so that **Run all** works without stopping. To type in your own diamond, set `USE_INTERACTIVE_INPUT = True` and run the cell again.

The same prediction is available in the **deployed Streamlit web app** (link at the top of this notebook).

In [ ]:
USE_INTERACTIVE_INPUT = False

example_diamond = {
    "carat": 1.01,
    "depth": 61.8,
    "table": 57.0,
    "x": 6.42,
    "y": 6.45,
    "z": 3.98,
    "cut": "Ideal",
    "color": "G",
    "clarity": "VS2",
}

def get_diamond_input():
    print("Enter Diamond Attributes (ranges seen in the data are shown):")
    diamond = {}
    for feature in numeric_features:
        low, high = X[feature].min(), X[feature].max()
        diamond[feature] = float(input(f"  {feature} ({low:.2f} - {high:.2f}): "))
    for feature in categorical_features:
        options = model_info["categories"][feature]
        diamond[feature] = input(f"  {feature} {options}: ").strip()
    return diamond

user_input = get_diamond_input() if USE_INTERACTIVE_INPUT else example_diamond

print("Diamond attributes:")
for feature, value in user_input.items():
    print(f"  {feature:<8}: {value}")

## Step 11.2: Convert User Input into Feature Vector

The feature vector must have **exactly the same columns, in the same order,** as the training data. The saved pipeline scales and one-hot encodes it automatically.

In [ ]:
feature_vector = pd.DataFrame([user_input])[numeric_features + categorical_features]
feature_vector

## Step 11.3: Load the Saved Model

In [ ]:
with open(MODEL_FILE, "rb") as f:
    loaded_model = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_model.named_steps['regressor'].__class__.__name__}")

## Step 11.4: Model Prediction

In [ ]:
predicted_price = loaded_model.predict(feature_vector)[0]

print(f"Predicted Price: ${predicted_price:,.2f}")
print(f"Typical error (cross-validation of {best_algorithm_name}): "
      f"RMSE about ${best_mean_rmse:,.0f}, MAE about ${best_mae:,.0f}")

# Step 12: Model Deployment (Streamlit Web App)

The saved model is served by a **Streamlit** web application (`app.py` in the submission folder), deployed on **Streamlit Community Cloud**:

```
GitHub repository                 Streamlit Community Cloud                 User's browser
├── app.py              ──deploy──►  installs requirements.txt   ◄──HTTPS──  enters carat, cut,
├── diamond_price_model.pkl          runs `streamlit run app.py`             colour, clarity, size
├── model_info.json                  loads the pickled pipeline              → predicted price
└── requirements.txt
```

The app loads `diamond_price_model.pkl` once (cached), lets the user enter the 9 attributes with sliders and drop-down lists, and shows the predicted price together with the cross-validation error and the comparison of all 5 algorithms. The step-by-step deployment instructions are in `README.md`.

**Live app:** see the link at the very top of this notebook.

# Step 13: Execute the Feedback Phase

After deployment, feedback from users and domain experts (e.g. jewellers) is used to improve the model. Potential improvements:

1. **Predict the logarithm of the price**: errors on expensive diamonds dominate the RMSE; training on `log(price)` treats a 10% error the same for cheap and expensive stones, which especially helps Ridge Regression.
2. **Tune hyper-parameters** with cross validation (`GridSearchCV` / `RandomizedSearchCV`), e.g. the learning rate and number of iterations of Hist Gradient Boosting, or the layers of the MLP.
3. **Use ordinal encoding** for `cut`, `color` and `clarity`, because they have a natural order (Fair < Good < … < Ideal).
4. **Add newer data**: diamond prices change over time, and this dataset is a historical snapshot.
5. **Show a prediction range** in the app (e.g. quantile regression), not just a single price.

# Conclusion

In this notebook we compared **5 regression algorithms** (none of them used in the Car Price notebook) on the **full Kaggle Diamonds dataset (53,794 unique diamonds)** using **10-fold cross validation** with the same folds, the same pre-processing pipeline and the same measures (RMSE, MAE, R²).

| Rank | Algorithm | Mean RMSE | Mean MAE | R² | Time (10 folds, our run) |
|---|---|---|---|---|---|
| 1 | Extra Trees | **\$543** | **\$266** | **0.981** | ≈ 60 s |
| 2 | Hist Gradient Boosting | \$551 | \$290 | 0.981 | ≈ 10 s |
| 3 | MLP Neural Network | \$682 | \$329 | 0.969 | ≈ 265 s |
| 4 | Ridge Regression | \$1,136 | \$735 | 0.919 | 2 s |
| 5 | AdaBoost | \$1,403 | \$1,028 | 0.876 | ≈ 45 s |

- **Extra Trees** was the most accurate algorithm, with **Hist Gradient Boosting** a very close second.
- For **deployment**, Extra Trees' final model is ≈ **680 MB**, far above GitHub's 100 MB limit and too large for a 1 GB Streamlit Cloud app. **Hist Gradient Boosting** was therefore deployed: its model is ≈ **0.4 MB** (≈ 1,800× smaller), it trains ≈ 6× faster, and its error is only ≈ 1.5% higher.
- **Linear (Ridge) and weak boosting (AdaBoost) models** performed worst, because diamond prices depend on the features in a strongly non-linear way.
- **More computing power is not automatically faster**: a GPU does not help scikit-learn unless a GPU library (cuML) supports the specific algorithm.
- The selected model is served by a **Streamlit web app** (link at the top of this notebook).